# Module 18 - Tool use

Use this notebook after `tests/test_tools.py` is passing. The notebook starts with deterministic local checks for schemas, parsers, dispatch, and the tool loop, then moves into optional ProdLM runs where a real local model has to choose when to call tools.

The deliverable is a tool-use postmortem: what tools you exposed, where the model used them correctly, where it failed, and what you would improve before turning this into the Module 19 agent loop.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import subprocess
import sys

from IPython.display import Markdown, display

from g2c.inference import (
    Backend,
    BackendInfo,
    InferenceResult,
    is_thinking_model,
    load_selected_backend,
)
from g2c.notebook_extras.sampling import printable
from g2c.tools import (
    DEFAULT_SYSTEM,
    Tool,
    ToolCall,
    ToolRegistry,
    calculator_evaluate,
    dispatch_tool_call,
    format_tool_results,
    make_calculator,
    make_read_file,
    make_run_python,
    make_web_search,
    parse_tool_calls,
    render_tools_for_prompt,
    run_with_tools,
    validate_arguments,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the tool tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 18 TODOs in `g2c/tools/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_tools.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 18 tool tests are not passing yet."


........................................................................ [ 38%]
........................................................................ [ 76%]
.............................................                            [100%]



## Display helpers

In [3]:
def short(text: Any, limit: int = 180) -> str:
    rendered = printable(str(text)).replace("\n", "\\n")
    if len(rendered) <= limit:
        return rendered
    return rendered[: limit - 3] + "..."


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> str:
    def cell(value: Any) -> str:
        text = str(value).replace("|", "\\|").replace("\n", "<br>")
        return text

    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(cell(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])


def show_tool(tool: Tool) -> None:
    props = tool.parameters.get("properties", {})
    required = set(tool.parameters.get("required", []))
    rows = []
    for name, spec in props.items():
        rows.append(
            {
                "argument": name,
                "type": spec.get("type", ""),
                "required": "yes" if name in required else "no",
                "description": spec.get("description", ""),
            }
        )
    display(Markdown(f"### `{tool.name}`\n\n{tool.description}"))
    if rows:
        display(Markdown(markdown_table(rows, ["argument", "type", "required", "description"])))
    else:
        display(Markdown("No arguments."))


def show_registry(registry: ToolRegistry) -> None:
    display(Markdown(f"Registered tools: `{', '.join(registry.names())}`"))
    for tool in registry:
        show_tool(tool)


def show_tool_run(result) -> None:
    print("stopped:", result.stopped_reason)
    print("final answer:", printable(result.final_answer or "(none)"))
    print()
    rows = []
    for step_index, step in enumerate(result.steps, start=1):
        if not step.tool_calls:
            rows.append(
                {
                    "step": step_index,
                    "tool": "(none)",
                    "arguments": "",
                    "error": "",
                    "output": short(step.completion),
                }
            )
            continue
        for call, tool_result in zip(step.tool_calls, step.tool_results):
            rows.append(
                {
                    "step": step_index,
                    "tool": call.name,
                    "arguments": json.dumps(call.arguments),
                    "error": "yes" if tool_result.is_error else "no",
                    "output": short(tool_result.output),
                }
            )
    display(Markdown(markdown_table(rows, ["step", "tool", "arguments", "error", "output"])))


## Build a local tool registry

Start with the three tools that let an assistant leave pure text generation: a calculator, a sandboxed file reader, and a small Python runner. The registry is both the prompt source and the dispatch table.

In [4]:
scratch_dir = repo_root / "data" / "work" / "module18"
scratch_dir.mkdir(parents=True, exist_ok=True)
(scratch_dir / "numbers.txt").write_text("12\n19\n31\n44\n", encoding="utf-8")
(scratch_dir / "sales.csv").write_text(
    "item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n",
    encoding="utf-8",
)

registry = ToolRegistry(
    [
        make_calculator(),
        make_read_file(root=scratch_dir),
        make_run_python(timeout=5.0, cwd=scratch_dir),
        make_web_search(),
    ]
)
show_registry(registry)


Registered tools: `calculator, read_file, run_python, web_search`

### `calculator`

Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.

| argument | type | required | description |
| --- | --- | --- | --- |
| expression | string | yes | Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'. |

### `read_file`

Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncated server-side; the model does not control the length.

| argument | type | required | description |
| --- | --- | --- | --- |
| path | string | yes | Relative path under the project root. |

### `run_python`

Execute a Python snippet in a subprocess and return stdout. Use print() to emit results. Has a wall-clock timeout. Each call runs in a fresh subprocess: variables, imports, and state from previous calls do not persist. Do all setup in every call, or do everything in one call. To operate on a file, pass its name via the `file` argument: the resolved path is bound to a string variable named `data` inside your code. Reference `data` rather than writing the filename as a literal string. Example: {"file": "sales.csv", "code": "import pandas as pd; print(pd.read_csv(data).head())"}.

| argument | type | required | description |
| --- | --- | --- | --- |
| code | string | yes | Python source to run. |
| file | string | no | Optional. Relative path to a file. The resolved absolute path is bound to a string variable named `data` inside your code, so you can write `pd.read_csv(data)` instead of `pd.read_csv('sales.csv')`. |

### `web_search`

Search the web for the given query. (Stub by default — wire a real backend for real results.)

| argument | type | required | description |
| --- | --- | --- | --- |
| query | string | yes | Search query. |

The model sees a rendered version of that registry inside the system prompt. This is the contract it must follow to call tools.

In [5]:
print(render_tools_for_prompt(registry.tools))

Tools available:
- calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.
  parameters:
  {
    "type": "object",
    "properties": {
      "expression": {
        "type": "string",
        "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."
      }
    },
    "required": [
      "expression"
    ]
  }
- read_file: Read a UTF-8 text file from the project root and return its contents. The path is interpreted relative to a sandboxed root directory. Output is truncated server-side; the model does not control the length.
  parameters:
  {
    "type": "object",
    "properties": {
      "path": {
        "type": "string",
        "description": "Relative path under the project root."
      }
    },
    "required": [
      "path"
    ]
  }
- run_python: Execute a Python snippet in a subprocess and return stdout. Use print() to emit results. Has a wall-clock timeout. Each call runs in a f

## Exercise 1 - Validate arguments

The parser should not decide whether arguments are safe for a specific tool. Validation needs the tool schema, so it happens after parsing and before dispatch.

In [6]:
calculator = registry.get("calculator")

valid_args = {"expression": "(23 * 17) + 5"}
validated = validate_arguments(calculator, valid_args)
print("validated:", validated)

bad_argument_sets = [
    {},
    {"expr": "23 * 17"},
    {"expression": 391},
    {"expression": "23 * 17", "round": True},
]

for args in bad_argument_sets:
    try:
        validate_arguments(calculator, args)
    except Exception as exc:
        print(f"{args!r} -> {type(exc).__name__}: {exc}")

validated: {'expression': '(23 * 17) + 5'}
{} -> ToolError: missing required arguments: ['expression']
{'expr': '23 * 17'} -> ToolError: missing required arguments: ['expression']
{'expression': 391} -> ToolError: argument 'expression': expected string, got int
{'expression': '23 * 17', 'round': True} -> ToolError: unknown arguments: ['round']


## Exercise 2 - Parse tool calls

A model output is just text. The parser extracts every well-formed `<tool_call>...</tool_call>` block, parses the JSON body, checks the shape, and assigns a `call_id`.

In [7]:
model_text = (
    "I should use the calculator.\n"
    "<tool_call>\n"
    '{"name": "calculator", "arguments": {"expression": "23 * 17"}}\n'
    "</tool_call>\n"
)

calls = parse_tool_calls(model_text)
for call in calls:
    print(call)

malformed_examples = [
    "<tool_call>not json</tool_call>",
    '<tool_call>{"arguments": {}}</tool_call>',
    '<tool_call>{"name": "calculator", "arguments": []}</tool_call>',
]

for text in malformed_examples:
    print(short(text), "->", parse_tool_calls(text))


ToolCall(name='calculator', arguments={'expression': '23 * 17'}, call_id='call_0_8c52aef3')
<tool_call>not json</tool_call> -> []
<tool_call>{"arguments": {}}</tool_call> -> []
<tool_call>{"name": "calculator", "arguments": []}</tool_call> -> []


## Exercise 3 - Dispatch calls and surface errors as data

`dispatch_tool_call` turns every outcome into a `ToolResult`. Unknown tools, malformed arguments, and tool exceptions become `is_error=True` results instead of crashing the loop.

In [8]:
manual_calls = [
    ToolCall(name="calculator", arguments={"expression": "23 * 17"}, call_id="manual_ok"),
    ToolCall(name="calculator", arguments={"expr": "23 * 17"}, call_id="manual_bad_args"),
    ToolCall(name="unknown_tool", arguments={}, call_id="manual_unknown"),
]

results = [dispatch_tool_call(registry, call) for call in manual_calls]
for result in results:
    print(result)

print()
print(format_tool_results(results))

ToolResult(call_id='manual_ok', name='calculator', output='391', is_error=False)
ToolResult(call_id='manual_bad_args', name='calculator', output="missing required arguments: ['expression']", is_error=True)
ToolResult(call_id='manual_unknown', name='unknown_tool', output='"no tool named \'unknown_tool\'; registered: [\'calculator\', \'read_file\', \'run_python\', \'web_search\']"', is_error=True)

<tool_result name="calculator" id="manual_ok">
391
</tool_result>

<tool_error name="calculator" id="manual_bad_args">
missing required arguments: ['expression']
</tool_error>

<tool_error name="unknown_tool" id="manual_unknown">
"no tool named 'unknown_tool'; registered: ['calculator', 'read_file', 'run_python', 'web_search']"
</tool_error>


## Exercise 4 - Safe calculator behavior

The calculator is deliberately not `eval`. It parses Python expression syntax into an AST and only accepts a small arithmetic whitelist.

In [9]:
safe_expressions = [
    "1 + 2",
    "(2 + 3) * 4",
    "2 ** 10",
    "7 / 2",
    "7 // 2",
    "-5 + 3",
]

unsafe_expressions = [
    "os.system('ls')",
    "__import__('os').system('ls')",
    "open('/etc/passwd').read()",
    "True + 1",
    "'hello'",
]

for expression in safe_expressions:
    print(expression, "=>", calculator_evaluate(expression))

print()
for expression in unsafe_expressions:
    try:
        calculator_evaluate(expression)
    except Exception as exc:
        print(expression, "=>", type(exc).__name__, exc)

1 + 2 => 3
(2 + 3) * 4 => 20
2 ** 10 => 1024
7 / 2 => 3.5
7 // 2 => 3
-5 + 3 => -2

os.system('ls') => ToolError AST node not allowed: Call
__import__('os').system('ls') => ToolError AST node not allowed: Call
open('/etc/passwd').read() => ToolError AST node not allowed: Call
True + 1 => ToolError booleans are not allowed in expressions
'hello' => ToolError constant must be a number, got str


## Exercise 5 - Run the loop with a fake backend

Before involving a real model, use a deterministic backend that emits known completions. This isolates the loop contract: complete, parse, dispatch, feed back, repeat.

In [10]:
class FakeBackend(Backend):
    def __init__(self, completions: list[str], *, model_id: str = "fake-tools") -> None:
        self._completions = list(completions)
        self._info = BackendInfo(name="fake", model_id=model_id)
        self.calls: list[dict[str, Any]] = []

    @property
    def info(self) -> BackendInfo:
        return self._info

    def complete(
        self,
        prompt: str,
        *,
        max_new_tokens: int = 128,
        temperature: float = 1.0,
        top_k: int | None = None,
        top_p: float | None = None,
    ) -> InferenceResult:
        if not self._completions:
            raise RuntimeError("FakeBackend has no completions left")
        completion = self._completions.pop(0)
        self.calls.append(
            {
                "prompt": prompt,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
                "top_k": top_k,
                "top_p": top_p,
            }
        )
        return InferenceResult(
            prompt=prompt,
            completion=completion,
            prompt_tokens=len(prompt.split()),
            completion_tokens=len(completion.split()),
            latency_ms=1.0,
            backend=self._info,
        )

In [11]:
fake_backend = FakeBackend(
    [
        '<tool_call>{"name": "calculator", "arguments": {"expression": "23 * 17"}}</tool_call>',
        "23 * 17 = 391.",
    ]
)

fake_result = run_with_tools(
    fake_backend,
    registry,
    "What is 23 times 17?",
    max_steps=3,
    temperature=0.0,
)
show_tool_run(fake_result)

print("\nPrompt after tool feedback included:")
print(short(fake_backend.calls[-1]["prompt"], limit=900))


stopped: no_more_calls
final answer: 23 * 17 = 391.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expression": "23 * 17"} | no | 391 |
| 2 | (none) |  |  | 23 * 17 = 391. |


Prompt after tool feedback included:
You are a helpful assistant with access to tools. To call a tool, emit a <tool_call> block whose body is JSON of the form {"name": "<tool>", "arguments": {...}}. The arguments must match the tool's parameters schema exactly. After you emit the block, stop and wait for the result. When you have enough information to answer the user, give your final answer in plain text without any <tool_call> blocks.\n\nTools available:\n- calculator: Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.\n  parameters:\n  {\n    "type": "object",\n    "properties": {\n      "expression": {\n        "type": "string",\n        "description": "Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'."\n      }\n    },\n    "required": [\n      "expression"\n    ]\n  }\n- read_file: Read a UTF-8 text file from the project root and retu...


The fake backend can also model recovery from an error. The first call uses the wrong argument name, the error is fed back, and the next turn corrects it.

In [12]:
recovering_backend = FakeBackend(
    [
        '<tool_call>{"name": "calculator", "arguments": {"expr": "23 * 17"}}</tool_call>',
        '<tool_call>{"name": "calculator", "arguments": {"expression": "23 * 17"}}</tool_call>',
        "The answer is 391.",
    ],
    model_id="fake-recovery",
)

recovery_result = run_with_tools(
    recovering_backend,
    registry,
    "What is 23 times 17?",
    max_steps=4,
    temperature=0.0,
)
show_tool_run(recovery_result)

stopped: no_more_calls
final answer: The answer is 391.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expr": "23 * 17"} | yes | missing required arguments: ['expression'] |
| 2 | calculator | {"expression": "23 * 17"} | no | 391 |
| 3 | (none) |  |  | The answer is 391. |

## Exercise 6 - Load a live backend for tool use

The default live backend is ProdLM. To test a course-trained model instead, set `MODEL_SELECTION = "course"` for the strongest course artifact, or set it to a base artifact name like `"TinyLLM-30M"`; the loader will prefer `-DPO`, then `-SFT`, then the base artifact.

In [22]:
MODEL_SELECTION = "ProdLM"  # "ProdLM", "course", or an artifact base/name such as "TinyLLM-30M"
PRODLM_MODEL_ID = "qwen3:4b"  # optional Ollama tag override when MODEL_SELECTION == "ProdLM"
LIVE_DEVICE = "auto"
LIVE_TORCH_DTYPE = "float16"

live_backend = None
try:
    live_backend = load_selected_backend(
        MODEL_SELECTION,
        repo_root=repo_root,
        prodlm_model_id=PRODLM_MODEL_ID,
        device=LIVE_DEVICE,
        torch_dtype=LIVE_TORCH_DTYPE,
        required=False,
    )
    if live_backend is None:
        print("No live backend loaded. Run ./prodlm.sh or choose an available artifact.")
    else:
        print("loaded:", live_backend.info)
except Exception as exc:
    print(f"Live backend unavailable: {type(exc).__name__}: {exc}")

loaded: BackendInfo(name='prodlm', model_id='qwen3:4b', extra={'base_url': 'http://localhost:11434', 'configured_name': 'ProdLM'})


In [23]:
STRICT_TOOL_SYSTEM = (
    "You are a helpful assistant with access to tools. "
    "For arithmetic, exact file contents, and Python data tasks, call the available tool instead of guessing. "
    "Call one tool per turn, then wait for the result. "
    "If a tool result starts with '[run_python: exit N]' or otherwise indicates an error, "
    "the call failed: read the error message, fix the call, and try the tool again. "
    "Do not give a final answer based on a failed tool call. "
    "When the run_python `code` argument contains string literals, use SINGLE quotes (e.g. 'sales.csv') "
    "so the embedded quotes do not collide with the JSON string delimiters around the code itself."
)


USE_NATIVE_TOOLS = True  # set False to use the text-format <tool_call> path

# Disable Ollama's "thinking mode" for models that ship with it on by
# default (Qwen3, DeepSeek-R1, ...). Thinking mode burns the
# max_new_tokens budget inside <think> blocks before the model emits
# any tool call or visible reply, which presents as an empty `(none)`
# final answer. `is_thinking_model` recognizes the known prefixes;
# override by setting THINK_OVERRIDE to True/False if needed.
THINK_OVERRIDE: bool | None = None
if THINK_OVERRIDE is not None:
    THINK_SETTING = THINK_OVERRIDE
elif live_backend is not None and is_thinking_model(live_backend.info.model_id):
    THINK_SETTING = False
else:
    THINK_SETTING = None  # let the server pick


def run_live_tool_question(question: str, *, tools: ToolRegistry = registry, max_steps: int = 5):
    if live_backend is None:
        print("No live backend loaded.")
        return None
    try:
        result = run_with_tools(
            live_backend,
            tools,
            question,
            system=STRICT_TOOL_SYSTEM,
            max_steps=max_steps,
            max_new_tokens=1024,
            temperature=0.0,
            use_native_tools=USE_NATIVE_TOOLS,
            think=THINK_SETTING,
        )
    except Exception as exc:
        print(f"Live tool run failed: {type(exc).__name__}: {exc}")
        return None
    show_tool_run(result)
    return result


Start with arithmetic. You are looking for three things: whether the model calls `calculator`, whether the JSON validates, and whether it stops after receiving the result.

In [24]:
arithmetic_question = "What is (1847 * 29) - 138? Use the calculator."
arithmetic_result = run_live_tool_question(arithmetic_question, max_steps=3)

stopped: no_more_calls
final answer: Okay, let me see. The user asked for (1847 * 29) - 138 using the calculator. I called the calculator function with that expression. The tool response was 53425.

Wait, I should verify if that's correct. Let me do a quick check. 1847 multiplied by 29. Let's compute 1847 * 29.

First, 1847 * 30 would be 55,410. Then subtract 1847 to get 29. So 55,410 - 1,847 = 53,563. Then subtract 138: 53,563 - 138 = 53,425. Yes, that matches the tool response. So the answer is 53425.

The tool worked correctly. So the final answer is 53425.
</think>

The result of $(1847 \times 29) - 138$ is **53425**.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expression": "(1847 * 29) - 138"} | no | 53425 |
| 2 | (none) |  |  | Okay, let me see. The user asked for (1847 * 29) - 138 using the calculator. I called the calculator function with that expression. The tool response was 53425.\n\nWait, I shoul... |

## Exercise 7 - Read, compute, answer

Now require two capabilities: reading a local file and doing arithmetic over the contents. Depending on the model, it may use `read_file` first and then either `calculator` or `run_python`.

In [25]:
file_question = "The file numbers.txt contains one number per line. Read it and report the sum and mean."
file_result = run_live_tool_question(file_question, max_steps=5)

stopped: no_more_calls
final answer: Okay, let's see. The user tried to run the Python code to read numbers.txt and calculate sum and mean, but got an error. The error message says: "ValueError: could not convert string to float: '/Users/colkitt/sith/toys/courses/g2c/data/work/module18/numbers.txt'".

Hmm, that's odd. Wait, the error message shows that the code is trying to convert a string to a float, but the string is the path to the file. Wait, no. Wait, the error says "could not convert string to float: '/Users/.../numbers.txt'". That suggests that the code is trying to read a line from the file, but the line is the path itself? Wait, no.

Wait, the problem is that the code uses `data.splitlines()`, but `data` is the entire file content. Wait, no. Wait, the error message says that the line being converted to float is the path to the file. Wait, maybe the code is not correctly handling the file path.

Wait, the error message shows that the code is trying to convert a string that's t

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | run_python | {"file": "numbers.txt", "code": "numbers = [float(line.strip()) for line in data.splitlines()]; total = sum(numbers); mean = total / len(numbers) if len(numbers) > 0 else 0; print(f'Sum: {total}, Mean: {mean}')"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "<string>", line 2, in <module>\n  File "<string>", line 2, in <listcomp>\nValueError: could not convert string ... |
| 2 | (none) |  |  | Okay, let's see. The user tried to run the Python code to read numbers.txt and calculate sum and mean, but got an error. The error message says: "ValueError: could not convert s... |

## Exercise 8 - Python as a data tool

`run_python` is powerful, but it is not a real sandbox. In this local course setting it is useful for small data tasks; in production it would need much stronger isolation.

In [26]:
python_question = "Read sales.csv, compute total revenue as units times price for each row, and report the total."
python_result = run_live_tool_question(python_question, max_steps=5)

stopped: no_more_calls
final answer: Okay, let's see. The user wants me to read sales.csv, compute total revenue as units times price for each row, and then report the total.

First, I need to figure out which tool to use. The available tools are calculator, read_file, run_python, and web_search. Since this involves processing a CSV file and doing calculations, the run_python function seems appropriate here. The read_file tool would just read the file, but the user wants to compute something, so running Python code makes sense.

The run_python function allows specifying a file path and code. The example given shows that if you pass a file, the resolved path is bound to a variable called `data`. So I can use that to read the CSV and process it.

Wait, the user says "compute total revenue as units times price for each row". So each row has units and price columns. I need to multiply those two columns and sum them up.

In Python, using pandas would be efficient here. The example in the to

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | (none) |  |  | Okay, let's see. The user wants me to read sales.csv, compute total revenue as units times price for each row, and then report the total.\n\nFirst, I need to figure out which to... |

## Exercise 9 - Add a custom tool

A tool is just a callable plus a schema. This small custom tool reverses text; replace it with a tool that would be useful in your own workflow.

In [27]:
def make_reverse_text() -> Tool:
    def _func(text: str) -> str:
        return text[::-1]

    return Tool(
        name="reverse_text",
        description="Reverse the characters in a string.",
        parameters={
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "Text to reverse."},
            },
            "required": ["text"],
        },
        func=_func,
    )

custom_registry = ToolRegistry([make_reverse_text(), make_calculator()])
show_registry(custom_registry)

custom_call = ToolCall(
    name="reverse_text",
    arguments={"text": "tool use"},
    call_id="custom_0",
)
print(dispatch_tool_call(custom_registry, custom_call))

Registered tools: `reverse_text, calculator`

### `reverse_text`

Reverse the characters in a string.

| argument | type | required | description |
| --- | --- | --- | --- |
| text | string | yes | Text to reverse. |

### `calculator`

Evaluate an arithmetic expression. Supports + - * / % ** // and parentheses. Returns the numeric result as a string.

| argument | type | required | description |
| --- | --- | --- | --- |
| expression | string | yes | Arithmetic expression to evaluate, e.g. '2 + 2' or '(3 ** 4) / 7'. |

ToolResult(call_id='custom_0', name='reverse_text', output='esu loot', is_error=False)


In [28]:
custom_question = "Reverse the text 'agent loop', then tell me what 14 * 19 is."
custom_result = run_live_tool_question(custom_question, tools=custom_registry, max_steps=5)

stopped: no_more_calls
final answer: Okay, let me go through the steps here. The user asked two things: reverse the text 'agent loop' and then compute 14 multiplied by 19.

First, I called the reverse_text function with the text 'agent loop'. The response was 'pool tnega'. Let me check that again. The original string is 'agent loop' (with a space). Reversing each character would start from the end. So the last character is 'p' (from 'loop'), then 'o', 'o', 'l', space, 't' (from 'agent'), 'n', 'e', 'g', 'a'. So the reversed string is 'pool tnega' (with a space between 'pool' and 'tnega'). That matches the tool response.

Next, I called the calculator function with the expression '14 * 19'. The tool response was 266. Let me verify that. 14 multiplied by 19: 14*20 is 280, minus 14 is 266. Yes, that's correct.

So the user's two requests have been handled. The reversed text is 'pool tnega' and the multiplication result is 266. Now I need to present both results in a clear answer.

Wait, th

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | reverse_text | {"text": "agent loop"} | no | pool tnega |
| 2 | calculator | {"expression": "14 * 19"} | no | 266 |
| 3 | (none) |  |  | Okay, let me go through the steps here. The user asked two things: reverse the text 'agent loop' and then compute 14 multiplied by 19.\n\nFirst, I called the reverse_text functi... |

## Exercise 10 - Toolformer-style ablation

Run the same question with and without tools. The important comparison is not whether ProdLM can do the task unaided once. It is whether the tool path is more reliable on exact arithmetic, file contents, and data transformation.

In [29]:
ablation_question = "What is 8437 * 29? Give only the final integer."

if live_backend is None:
    print("No live backend loaded.")
else:
    try:
        direct = live_backend.complete(
            ablation_question,
            max_new_tokens=80,
            temperature=0.0,
        )
        print("direct completion:")
        print(printable(direct.completion))
        print("\nwith tools:")
        ablation_result = run_live_tool_question(ablation_question, tools=ToolRegistry([make_calculator()]), max_steps=3)
    except Exception as exc:
        print(f"ablation failed: {type(exc).__name__}: {exc}")


direct completion:


with tools:
stopped: no_more_calls
final answer: Okay, let's see. The user is asking for the result of 8437 multiplied by 29. They want just the final integer.

First, I need to check if there's a tool available to handle this arithmetic. The tools provided include a calculator function that evaluates arithmetic expressions. The function takes an expression as a string. So I should use that.

The expression here is "8437 * 29". I need to make sure the syntax is correct. The calculator supports * for multiplication, so that should work.

Wait, the tool's description says it supports + - * / % ** // and parentheses. So the expression should be written with a space or without? The example given is '2 + 2' or '(3 ** 4) / 7', so probably the expression can have spaces or not. But in the code, it's better to write it without spaces to avoid any issues. Let me check.

The user wrote "8437 * 29", so the expression in the tool would be "8437 * 29". But maybe the tool expect

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | (none) |  |  | Okay, let's see. The user is asking for the result of 8437 multiplied by 29. They want just the final integer.\n\nFirst, I need to check if there's a tool available to handle th... |

## Exercise 11 - Short evaluation session

Use this cell to collect a small transcript for your postmortem. Keep the question list mixed: some should require tools, and some should not.

Requires a live backend; the cell prints a friendly skip message if none is loaded.

In [30]:
session_questions = [
    "What is 511 * 37?",
    "Read numbers.txt and report the largest number.",
    "Read sales.csv and compute total revenue.",
    "What is the capital of France? Do not use tools unless needed.",
    "Try to read missing.txt and then explain what happened.",
]

transcript = []
if live_backend is None:
    print("No live backend loaded.")
else:
    for question in session_questions:
        print("=" * 80)
        print(question)
        result = run_live_tool_question(question, max_steps=5)
        transcript.append(result)

What is 511 * 37?
stopped: no_more_calls
final answer: Okay, let me check the user's question again. They asked, "What is 511 * 37?" I used the calculator tool with the expression "511 * 37" and got the result 18907.

Wait, I should verify if that's correct. Let me do a quick mental calculation. 500 * 37 is 18,500. Then 11 * 37 is 407. Adding those together: 18,500 + 407 = 18,907. Yep, that matches the result from the calculator. So the answer is correct.

The tool response was 18907, which is the right answer. So I can just present that to the user.
</think>

The result of $ 511 \times 37 $ is **18907**.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | calculator | {"expression": "511 * 37"} | no | 18907 |
| 2 | (none) |  |  | Okay, let me check the user's question again. They asked, "What is 511 * 37?" I used the calculator tool with the expression "511 * 37" and got the result 18907.\n\nWait, I shou... |

Read numbers.txt and report the largest number.
stopped: no_more_calls
final answer: Okay, let's see. The user wants me to read numbers.txt and report the largest number. Hmm, first I need to figure out which tool to use here.

Looking at the available tools: there's read_file which reads a UTF-8 text file from the project root. That seems relevant here because the user is asking to read numbers.txt. So I should call read_file with the path "numbers.txt".

Wait, the read_file function returns the contents of the file. Once I have the contents, I need to process it to find the largest number. But the user says "report the largest number", so after reading the file, I have to parse the numbers and find the max.

But the tools provided don't include a function to process the numbers. The run_python function can execute Python code. Maybe I should use run_python to read the file and then process it. Wait, the read_file tool gives me the file contents as a string. Let me think.

Wait, the p

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | (none) |  |  | Okay, let's see. The user wants me to read numbers.txt and report the largest number. Hmm, first I need to figure out which tool to use here.\n\nLooking at the available tools: ... |

Read sales.csv and compute total revenue.
stopped: no_more_calls
final answer: Okay, let's see. The user wants to compute total revenue from sales.csv. The first attempt used run_python with a code that tried to sum the 'revenue' column, but that failed because the CSV doesn't have a 'revenue' column. Then I called read_file to get the contents of sales.csv, which shows the data has columns: item, units, price.

So the CSV has three columns: item (like notebook, pen, marker), units (how many of each item sold), and price (the price per unit). Revenue per item would be units multiplied by price. So total revenue is the sum of (units * price) for all rows.

I need to write a Python code that reads the CSV, calculates revenue per row as units * price, then sums them up.

The previous run_python call used 'revenue' column which doesn't exist, so now I should adjust the code to compute revenue on the fly.

Let me structure the code:

1. Read the CSV into a pandas DataFrame.
2. For each row,

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | run_python | {"file": "sales.csv", "code": "import pandas as pd; df = pd.read_csv(data); print(df['revenue'].sum())"} | no | [run_python: exit 1]\nTraceback (most recent call last):\n  File "/Users/colkitt/sith/toys/courses/g2c/.venv/lib/python3.11/site-packages/pandas/core/indexes/base.py", line 3641... |
| 2 | read_file | {"path": "sales.csv"} | no | item,units,price\nnotebook,3,7.50\npen,12,1.25\nmarker,5,2.00\n |
| 3 | (none) |  |  | Okay, let's see. The user wants to compute total revenue from sales.csv. The first attempt used run_python with a code that tried to sum the 'revenue' column, but that failed be... |

What is the capital of France? Do not use tools unless needed.
stopped: no_more_calls
final answer: Okay, the user is asking for the capital of France. Let me think. I know that Paris is the capital of France. But wait, the instructions say not to use tools unless needed. The tools provided are for arithmetic, reading files, running Python, and web search. Since this is a straightforward fact that I already know, I don't need to call any tools. The user said not to use tools unless needed, so I should just answer directly.

Wait, but maybe I should check if there's a possibility that the user expects me to use a tool. But the capital of France is a well-known fact. The tools here include a web search, but since I know it's Paris, I don't need to do that. The problem says "Do not use tools unless needed." So in this case, no tools are needed. I should respond with Paris without using any tool calls.
</think>

The capital of France is Paris.



| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | (none) |  |  | Okay, the user is asking for the capital of France. Let me think. I know that Paris is the capital of France. But wait, the instructions say not to use tools unless needed. The ... |

Try to read missing.txt and then explain what happened.
stopped: no_more_calls
final answer: Okay, let me see. The user asked to try reading missing.txt and then explain what happened. I called the read_file function with the path "missing.txt". The response was "file not found: 'missing.txt'".

So the explanation here is that the file missing.txt doesn't exist in the project root. The error message from the tool clearly states that the file wasn't found. I need to tell the user that the file is missing and maybe suggest checking the file path or if it's supposed to be there.

Wait, the user might have intended a different file name, but since they specified missing.txt, the answer is straightforward. The explanation should state that the file wasn't found, so the read operation failed. No need for extra steps here. Just relay the error message as the result.

I should phrase it clearly: The file 'missing.txt' does not exist in the project root. Therefore, the read operation failed wit

| step | tool | arguments | error | output |
| --- | --- | --- | --- | --- |
| 1 | read_file | {"path": "missing.txt"} | yes | file not found: 'missing.txt' |
| 2 | (none) |  |  | Okay, let me see. The user asked to try reading missing.txt and then explain what happened. I called the read_file function with the path "missing.txt". The response was "file n... |

## Postmortem notes

Write `docs/tools-postmortem.md` in 3-4 paragraphs. Cover:

- What you wired up: model, tools, max step budget, and system prompt.
- What worked: tasks where the model reliably called the right tool.
- Where it broke: malformed JSON, wrong tool, missing tool, looping, or stopping too early.
- What you would improve before Module 19: ReAct formatting, better stop criteria, stricter parsing, more tool-specific examples, or richer evals.